In [ ]:
import subprocess
import os

# Images to use
use_images = [0, 4, 9]

mainpath='/data_fast/Work/MS_HF_cohort/images/figures/HRO2/c1_tracked_all_with_cylinders/'

# Create temporary trimmed images
trimmed_files = []
for idx in use_images:
    input_file = f"{mainpath}image.{idx:02d}.png"
    trimmed_file = f"{mainpath}trimmed_{idx:02d}.png"
    subprocess.run([
        'convert', input_file, '-trim', trimmed_file
    ], check=True)
    trimmed_files.append(trimmed_file)

# Append images horizontally
subprocess.run([
    'convert',
    *trimmed_files,
    '+append',
    f'{mainpath}temp_stack.png'
], check=True)

# Get dimensions of the stacked image
result = subprocess.run([
    'identify', '-format', '%w %h', f'{mainpath}temp_stack.png'
], capture_output=True, text=True, check=True)
width, height = map(int, result.stdout.strip().split())

# Add arrow and label
arrow_y = height + 40  # Position below the image
subprocess.run([
    'convert', f'{mainpath}temp_stack.png',
    '-gravity', 'North',
    '-background', 'white',
    '-splice', '0x80',  # Add space below
    '-fill', 'black',
    '-stroke', 'black',
    '-strokewidth', '2',
    '-draw', f'line 20,{arrow_y} {width-20},{arrow_y}',  # Horizontal line
    '-draw', f'polygon {width-20},{arrow_y} {width-30},{arrow_y-5} {width-30},{arrow_y+5}',  # Arrowhead
    '-pointsize', '24',
    '-annotate', f'+0+{arrow_y + 30}', 'Cardiac Cycle',
    f'{mainpath}cine_stack.png'
], check=True)

# Cleanup temporary files
os.remove(f'{mainpath}temp_stack.png')
for f in trimmed_files:
    os.remove(f)

print(f"Created {mainpath}cine_stack.png with cardiac cycle arrow")